# C. elegans Gene–Phenotype (WormBase) — Relation-Wise KG Triple Construction

## Purpose

This notebook implements the **complete end-to-end pipeline** for constructing Gene–Phenotype Knowledge Graph (KG) triples for *C. elegans*. It processes raw WormBase phenotype data, maps phenotype ontology IDs to names, maps C. elegans genes to human orthologs, enriches with NCBI gene descriptions, and produces standardized relation-wise KG triples.



## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `phenotype_ontology.WS292.obo` | WormBase | Phenotype ontology OBO file (ID → name mapping) |
| `phenotype_association.WS292.wb` | WormBase | Gene–phenotype association file |
| `Caenorhabditis_elegans.gene_info` | NCBI Gene | Gene annotations for C. elegans |

---
## Setup

Import required libraries and define all base paths.

In [1]:
import pandas as pd
import numpy as np

In [13]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================


your_path_here =  '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

# WormBase phenotype ontology OBO file
PHENOTYPE_OBO_PATH = f'{your_path_here}/wormbase/phenotype_ontology.WS292.obo'

# WormBase phenotype association file
PHENOTYPE_ASSOC_PATH = f'{your_path_here}/wormbase/phenotype_association.WS292.wb'

# gProfiler human-C.elegans ortholog mapping
# GPROFILER_ORTHO_PATH = f'{your_path_here}/wormbase/gProfiler_hsapiens_celegans_6-8-2024_8-20-59 AM.csv'

# NCBI gene info for C. elegans
NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info'

# Intermediate CSV (saved for record-keeping)
# INTERMEDIATE_CSV_PATH = 'Cele_Gene_ortho_Phenotype.csv'

# Final output path
OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/wormbase/wormbase_CELE_GENE_PHENOTYPE.csv'

---
## 1. Parse Phenotype Ontology (OBO → ID→Name Dictionary)

Parse the WormBase phenotype ontology OBO file to build a dictionary mapping phenotype IDs (e.g., `WBPhenotype:0000000`) to human-readable names (e.g., `chromosome instability`).

In [4]:
# =============================================================================
# Parse WormBase phenotype ontology OBO file
# =============================================================================
def parse_obo(file_path):
    """Parse an OBO ontology file into a pandas DataFrame."""
    with open(file_path, 'r') as file:
        lines = file.readlines()

    data = []
    current_term = {}
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line == '[Term]':
            if current_term:
                data.append(current_term)
            current_term = {}
        else:
            key, _, value = line.partition(': ')
            if key in current_term:
                if not isinstance(current_term[key], list):
                    current_term[key] = [current_term[key]]
                current_term[key].append(value)
            else:
                current_term[key] = value

    if current_term:
        data.append(current_term)

    return pd.DataFrame(data)

# Parse the OBO file
phenotype_ontology_df = parse_obo(PHENOTYPE_OBO_PATH)

# Ensure 'id' and 'name' columns contain hashable types (some may be lists)
phenotype_ontology_df['id'] = phenotype_ontology_df['id'].apply(lambda x: str(x) if isinstance(x, list) else x)
phenotype_ontology_df['name'] = phenotype_ontology_df['name'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Build phenotype ID → name dictionary
phenotype_id_to_name = phenotype_ontology_df.set_index('id')['name'].to_dict()

print(f"Phenotype ontology terms loaded: {len(phenotype_ontology_df):,}")
print(f"Phenotype ID → Name dictionary size: {len(phenotype_id_to_name):,}")

Phenotype ontology terms loaded: 2,698
Phenotype ID → Name dictionary size: 2,698


In [19]:
NCBI_Cele_gene = pd.read_csv(NCBI_GENE_INFO_PATH,sep = '\t')
NCBI_Cele_gene
NCBI_Cele_gene['LocusTag'] = NCBI_Cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

NCBI_Cele_gene_Locus_2_GeneSymd_ict = dict(zip(NCBI_Cele_gene['Symbol'], NCBI_Cele_gene['LocusTag']))
NCBI_Cele_gene_Locus_2_GeneSymd_ict
NCBI_Cele_Locustag_Descrip_dict = dict(zip(NCBI_Cele_gene['Symbol'], NCBI_Cele_gene['description']))
NCBI_Cele_Locustag_Descrip_dict

{'homt-1': 'Alpha N-terminal protein methyltransferase 1',
 'nlp-40': 'Peptide P4',
 'rcor-1': 'REST corepressor rcor-1',
 'sesn-1': 'Sestrin homolog',
 'pgs-1': 'CDP-diacylglycerol--glycerol-3-phosphate 3-phosphatidyltransferase',
 'Y48G1C.5': 'Tyrosine-protein phosphatase domain-containing protein',
 'Y48G1C.6': 'HTH CENPB-type domain-containing protein',
 'pid-2': 'Protein pid-2',
 'rab-11.1': 'Ras-related protein rab-11.1',
 'rpl-7': 'Large ribosomal subunit protein uL30',
 'F53G12.9': 'GST_C_6 domain-containing protein;Metaxin glutathione S-transferase domain-containing protein',
 'F53G12.8': 'Uncharacterized protein',
 'col-45': 'Collagen triple helix repeat protein;Nematode cuticle collagen N-terminal domain-containing protein',
 'spe-8': 'Spermatocyte protein spe-8',
 'mex-3': 'Growth-regulating factor;K Homology domain-containing protein',
 'bli-3': 'Dual oxidase 1',
 'ptr-11': 'SSD domain-containing protein',
 'cest-27': 'Carboxylic ester hydrolase',
 'F56C11.3': 'Sulfhydryl 

In [18]:
NCBI_Cele_gene_Locus_2_GeneSymd_ict
NCBI_Cele_gene

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,6239,171590,homt-1,Y74C9A.3,-,WormBase:WBGene00022277|AllianceGenome:WB:WBGe...,I,-,Alpha N-terminal protein methyltransferase 1,protein-coding,homt-1,Alpha N-terminal protein methyltransferase 1,O,Alpha N-terminal protein methyltransferase 1,20250308,-
1,6239,171591,nlp-40,Y74C9A.2,-,WormBase:WBGene00022276|AllianceGenome:WB:WBGe...,I,-,Peptide P4,protein-coding,nlp-40,Peptide P4,O,Peptide P4,20250308,-
2,6239,171592,rcor-1,Y74C9A.4,-,WormBase:WBGene00022278|AllianceGenome:WB:WBGe...,I,-,REST corepressor rcor-1,protein-coding,rcor-1,REST corepressor rcor-1,O,REST corepressor rcor-1,20250308,-
3,6239,171593,sesn-1,Y74C9A.5,-,WormBase:WBGene00022279|AllianceGenome:WB:WBGe...,I,-,Sestrin homolog,protein-coding,sesn-1,Sestrin homolog,O,Sestrin homolog,20250308,-
4,6239,171594,pgs-1,Y48G1C.4,-,WormBase:WBGene00021677|AllianceGenome:WB:WBGe...,I,-,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,protein-coding,pgs-1,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,O,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,20250308,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46922,6239,80510347,azyx-1,F42G4.11,-,WormBase:WBGene00306133|AllianceGenome:WB:WBGe...,II,-,Alternative N-terminal open reading frame of Z...,protein-coding,azyx-1,Alternative N-terminal open reading frame of Z...,O,Alternative N-terminal open reading frame of Z...,20250308,-
46923,6239,80510348,F54D10.10,F54D10.10,-,WormBase:WBGene00306132|AllianceGenome:WB:WBGe...,II,-,Uncharacterized protein,protein-coding,F54D10.10,Uncharacterized protein,O,Uncharacterized protein,20241028,-
46924,6239,80510349,F58B6.11,F58B6.11,-,WormBase:WBGene00306125|AllianceGenome:WB:WBGe...,III,-,Uncharacterized protein,protein-coding,F58B6.11,Uncharacterized protein,O,Uncharacterized protein,20241028,-
46925,6239,80510350,Y34B4A.20,Y34B4A.20,-,WormBase:WBGene00306131|AllianceGenome:WB:WBGe...,X,-,Uncharacterized protein,protein-coding,Y34B4A.20,Uncharacterized protein,O,Uncharacterized protein,20241028,-


---
## 2. WormBase Phenotype Association Processing

Load the WormBase phenotype association file, map phenotype IDs to names using the ontology dictionary, select relevant columns, and deduplicate.

In [23]:
# =============================================================================
# Load and process WormBase phenotype associations
# =============================================================================

# Load phenotype association file (tab-separated, no header)
gene_pheno_raw = pd.read_csv(PHENOTYPE_ASSOC_PATH, sep='\t', header=None)

# Rename key columns
gene_pheno_raw.rename(columns={1: 'Gene_id', 2: 'Gene_name'}, inplace=True)

# Map phenotype IDs (column 4) to phenotype names using ontology dictionary
gene_pheno_raw['Phenotype'] = gene_pheno_raw[4].map(phenotype_id_to_name)

# Select Gene_name and Phenotype, deduplicate
gene_pheno_raw = gene_pheno_raw[['Gene_name', 'Phenotype']]
gene_pheno_raw = gene_pheno_raw.drop_duplicates()

print(f"Gene-Phenotype associations after deduplication: {len(gene_pheno_raw):,}")
gene_pheno_raw.head()

Gene-Phenotype associations after deduplication: 258,940


,Gene_name,Phenotype
0,aap-1,extended life span
2,aap-1,thermotolerance increased
3,aap-1,dauer formation variant
4,aap-1,slow development
5,aap-1,DMPP resistant


In [24]:
# =============================================================================
# Format as triples
# =============================================================================

# Note: DataFrame.rename() with inplace=True is planned for deprecation in future pandas
gene_pheno_raw.rename(columns={'Gene_name': 'Head', 'Phenotype': 'Tail'}, inplace=True)

gene_pheno_raw['Head_detail_name'] = gene_pheno_raw['Head'].map(NCBI_Cele_Locustag_Descrip_dict)

gene_pheno_raw = gene_pheno_raw[~gene_pheno_raw['Head_detail_name'].isna()]

gene_pheno_raw['Head_type'] = 'Gene'
gene_pheno_raw['Tail_type'] = 'Phenotype'
gene_pheno_raw['Relation'] = 'Gene_Phenotype'
gene_pheno_raw['Relation_source'] = 'wormbase'

# Reorder columns
gene_pheno_raw = gene_pheno_raw[['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type', 'Head_detail_name', 'Relation_source']]

print(f"Formatted triples: {len(gene_pheno_raw):,}")
gene_pheno_raw
gene_pheno_raw.to_csv(f'{OUTPUT_PATH}',index = None)
OUTPUT_PATH

Formatted triples: 253,469


'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/wormbase/wormbase_CELE_GENE_PHENOTYPE.csv'

In [25]:
gene_pheno_raw

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Relation_source
0,aap-1,Gene_Phenotype,extended life span,Gene,Phenotype,SH2 domain-containing protein,wormbase
2,aap-1,Gene_Phenotype,thermotolerance increased,Gene,Phenotype,SH2 domain-containing protein,wormbase
3,aap-1,Gene_Phenotype,dauer formation variant,Gene,Phenotype,SH2 domain-containing protein,wormbase
4,aap-1,Gene_Phenotype,slow development,Gene,Phenotype,SH2 domain-containing protein,wormbase
5,aap-1,Gene_Phenotype,DMPP resistant,Gene,Phenotype,SH2 domain-containing protein,wormbase
...,...,...,...,...,...,...,...
438040,set-21,Gene_Phenotype,transgene expression reduced,Gene,Phenotype,SET domain-containing protein,wormbase
438041,set-21,Gene_Phenotype,transgene expression increased,Gene,Phenotype,SET domain-containing protein,wormbase
438042,dot-1.5,Gene_Phenotype,transgene expression reduced,Gene,Phenotype,"Histone-lysine N-methyltransferase, H3 lysine-...",wormbase
438043,lsy-12,Gene_Phenotype,transgene expression reduced,Gene,Phenotype,Histone acetyltransferase lsy-12,wormbase
